In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib pyyaml
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q soundfile librosa
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载DeepACE代码
import os
if not os.path.exists('DeepACE_torch'):
    print('请上传DeepACE_torch目录到Colab，或从Google Drive挂载')
    print('替代方案：将DeepACE_torch上传为zip文件')

# 生成mini数据集
if not os.path.exists('DeepACE_torch/data'):
    print('运行prepare_mini_dataset.py生成数据集...')
    # 如果有prepare_mini_dataset.py的话
    # !python scripts/prepare_mini_dataset.py
    print('注意：需要先上传代码和数据，或使用合成数据')


# 模块4 第3次课（可选）：DeepACE修改实验本notebook是模块4的可选深入内容，侧重于：1. 实验设计方法论2. 修改通道选择策略3. 修改损失函数4. 在不同噪声条件下评估> "改模型不是瞎改，要有假设。" 每个实验都要先明确：为什么要改？改了之后预期有什么效果？---

## §1 实验设计方法论一个好的修改实验需要：1. **假设 (Hypothesis)**：基于对模型和问题的理解，提出一个明确的假设   - 例："如果减少通道选择数N，模型会更关注最重要的频带，可能提高在噪声中的鲁棒性"2. **修改方案 (Modification)**：明确要改什么、怎么改   - 例："将config.yaml中的M从22改为12"3. **对比基线 (Baseline)**：用未修改的模型作为参照   - 例："M=22的原始模型在相同数据上训练的结果"4. **评估方法 (Evaluation)**：用什么指标来衡量效果   - 例："在不同SNR下的MSE，以及通道激活模式的变化"5. **结果分析 (Analysis)**：结果是否支持假设？为什么？> 实验模板：假设 -> 修改 -> 对比 -> 评估 -> 分析

In [ ]:
import torchimport torch.nn as nnimport yamlimport osimport sysimport numpy as npimport matplotlib.pyplot as plt# 添加路径DEEPACE_DIR = os.path.join('DeepACE_torch')sys.path.insert(0, DEEPACE_DIR)# 加载配置with open(os.path.join(DEEPACE_DIR, 'config.yaml'), 'r') as f:    config = yaml.safe_load(f)print("=== 原始配置 ===")model_cfg = config['DeepACE']for k, v in model_cfg.items():    print("  %s: %s" % (k, v))

## §2 实验1：修改通道选择策略### 假设ACE策略中，Nmaxima（每帧选取的最大通道数）是一个关键参数。传统ACE固定为8。我们探索：- 如果DeepACE的输出通道数M改变，模型如何适应？- 是否可以让模型学习"自适应通道选择"——不固定N，而是根据输入动态决定激活几个通道？### 修改方案**实验1a**: 改变输出通道数 M (22 -> 12, 16, 8)**实验1b**: 添加注意力机制实现自适应通道选择

In [ ]:
# === 实验1a: 不同输出通道数对比 ===from model import DeepACE# 创建不同 M 的模型configs_to_test = [    {"M": 22, "label": "M=22 (原始, 22电极)"},    {"M": 16, "label": "M=16 (16通道)"},    {"M": 12, "label": "M=12 (12通道)"},    {"M": 8,  "label": "M=8  (8通道)"},]models = {}for cfg in configs_to_test:    m = DeepACE(        L=model_cfg['L'], N=model_cfg['N'], P=model_cfg['P'],        B=model_cfg['B'], S=model_cfg['S'], H=model_cfg['H'],        X=model_cfg['X'], R=model_cfg['R'], M=cfg['M'],        msk_activate=model_cfg['msk_activate'], causal=model_cfg['causal']    )    models[cfg['M']] = m    n_params = sum(p.numel() for p in m.parameters())    print("%s -> 参数量: %.2fM" % (cfg['label'], n_params / 1e6))

In [ ]:
# 对比不同 M 的模型前向传播dummy_input = torch.randn(2, 1, 64000)print("不同M的模型输出形状对比:")print("-" * 40)for M, m in models.items():    with torch.no_grad():        output = m(dummy_input)    print("M=%2d -> 输出形状: %s" % (M, str(output.shape)))print()print("观察：M只影响输出通道数，不影响时间分辨率。")print("时间维度由Encoder的stride决定（与M无关）。")

### 实验1b: 添加通道注意力在Mask Generator的输出后，添加一个通道注意力模块，让模型学习"每个通道应该在什么条件下被激活"。

In [ ]:
class ChannelAttention(nn.Module):    """通道注意力模块：学习每个通道的重要性权重"""    def __init__(self, n_channels, reduction=4):        super().__init__()        self.avg_pool = nn.AdaptiveAvgPool1d(1)        self.fc = nn.Sequential(            nn.Linear(n_channels, n_channels // reduction),            nn.ReLU(),            nn.Linear(n_channels // reduction, n_channels),            nn.Sigmoid()        )    def forward(self, x):        # x: (batch, channels, time)        b, c, t = x.shape        y = self.avg_pool(x).view(b, c)  # (batch, channels)        y = self.fc(y).view(b, c, 1)     # (batch, channels, 1)        return x * y# 在DeepACE的基础上添加通道注意力class DeepACEWithAttention(nn.Module):    def __init__(self, base_model, n_channels=22, reduction=4):        super().__init__()        self.base = base_model        self.channel_attn = ChannelAttention(n_channels, reduction)    def forward(self, x):        out = self.base(x)        out = self.channel_attn(out)        return out# 创建带注意力的模型base_model = DeepACE(    L=model_cfg['L'], N=model_cfg['N'], P=model_cfg['P'],    B=model_cfg['B'], S=model_cfg['S'], H=model_cfg['H'],    X=model_cfg['X'], R=model_cfg['R'], M=22,    msk_activate=model_cfg['msk_activate'], causal=model_cfg['causal'])model_with_attn = DeepACEWithAttention(base_model, n_channels=22)n_params_base = sum(p.numel() for p in base_model.parameters())n_params_attn = sum(p.numel() for p in model_with_attn.parameters())print("原始模型参数量: %.2fM" % (n_params_base / 1e6))print("带注意力模型参数量: %.2fM (增加%.2fK)" % (    n_params_attn / 1e6, (n_params_attn - n_params_base) / 1e3))# 测试前向传播dummy = torch.randn(1, 1, 64000)with torch.no_grad():    out = model_with_attn(dummy)print("带注意力模型输出:", out.shape)

## §3 实验2：修改损失函数### 假设MSE损失对所有通道和所有时间步一视同仁。但CI的电极通道有不同的频率范围和重要性——低频通道（基频信息）可能比高频通道对语音理解更重要。### 修改方案- **加权MSE**: 给不同通道分配不同权重- **感知损失**: 加入频谱约束- **稀疏性约束**: 鼓励模型只激活少量通道（模拟N-of-M）

In [ ]:
# === 修改损失函数实验 ===class WeightedChannelMSE(nn.Module):    """通道加权MSE损失"""    def __init__(self, n_channels=22):        super().__init__()        # 可学习的通道权重（初始化为均匀）        self.channel_weights = nn.Parameter(torch.ones(n_channels))    def forward(self, pred, target):        # pred, target: (batch, 22, T)        weights = torch.softmax(self.channel_weights, dim=0)  # 归一化        weights = weights.view(1, -1, 1)  # (1, 22, 1)        squared_error = (pred - target) ** 2        weighted_error = squared_error * weights        return weighted_error.mean()class SparseMSE(nn.Module):    """MSE + 稀疏性约束"""    def __init__(self, sparsity_weight=0.01):        super().__init__()        self.sparsity_weight = sparsity_weight        self.mse = nn.MSELoss()    def forward(self, pred, target):        mse_loss = self.mse(pred, target)        # L1稀疏性：鼓励输出中有更多零值        sparsity_loss = torch.mean(torch.abs(pred))        return mse_loss + self.sparsity_weight * sparsity_loss# 创建不同的损失函数losses = {    'MSE (baseline)': nn.MSELoss(),    'L1': nn.L1Loss(),    'WeightedChannelMSE': WeightedChannelMSE(22),    'SparseMSE (0.01)': SparseMSE(0.01),    'SparseMSE (0.1)': SparseMSE(0.1),}print("可用损失函数:")for name, loss_fn in losses.items():    n_params = sum(p.numel() for p in loss_fn.parameters()) if hasattr(loss_fn, 'parameters') else 0    print("  %s (额外参数: %d)" % (name, n_params))

In [ ]:
# 对比不同损失函数的行为torch.manual_seed(42)pred = torch.randn(2, 22, 100) * 0.5 + 0.5pred = torch.clamp(pred, 0, 1)target = torch.randn(2, 22, 100) * 0.3 + 0.5target = torch.clamp(target, 0, 1)print("不同损失函数的值:")print("-" * 40)for name, loss_fn in losses.items():    loss_val = loss_fn(pred, target)    print("  %s: %.6f" % (name, loss_val.item()))print()print("观察：")print("- L1 对离群值不如MSE敏感")print("- SparseMSE 额外惩罚了激活（鼓励稀疏性）")print("- WeightedChannelMSE 的权重是可学习的，训练过程中会自动调整")

## §4 实验3：在不同噪声条件下评估### 假设模型在不同SNR下的性能不同。我们期望：SNR越高，DeepACE的输出越接近target；SNR越低，误差越大。但关键问题是：DeepACE相比传统ACE，在低SNR下的优势是否更大？

In [ ]:
# 生成不同SNR的测试数据def generate_test_signal(duration=4.0, sr=16000, snr_db=10):    """生成指定SNR的测试信号"""    n_samples = int(sr * duration)    t = np.linspace(0, duration, n_samples, endpoint=False)    # 简单语音信号    signal = np.zeros_like(t)    for h in range(1, 6):        signal += (0.3 / h) * np.sin(2 * np.pi * 150 * h * t)    signal = signal / np.max(np.abs(signal)) * 0.8    # 噪声    noise = np.random.randn(n_samples)    # 按SNR混合    speech_power = np.mean(signal ** 2)    noise_power = np.mean(noise ** 2)    snr_linear = 10 ** (snr_db / 10)    noise_scaled = noise * np.sqrt(speech_power / (noise_power * snr_linear + 1e-8))    mixture = signal + noise_scaled    mixture = mixture / np.max(np.abs(mixture)) * 0.9    return mixture.astype(np.float32)# 不同SNR测试snr_levels = [-10, -5, 0, 5, 10, 15, 20]print("生成不同SNR的测试信号:")for snr in snr_levels:    mix = generate_test_signal(snr_db=snr)    print("  SNR=%3ddB: shape=%s, range=[%.3f, %.3f]" % (        snr, mix.shape, mix.min(), mix.max()))

In [ ]:
# 用原始模型测试不同SNRfrom model import DeepACE# 加载原始模型model = DeepACE(    L=model_cfg['L'], N=model_cfg['N'], P=model_cfg['P'],    B=model_cfg['B'], S=model_cfg['S'], H=model_cfg['H'],    X=model_cfg['X'], R=model_cfg['R'], M=model_cfg['M'],    msk_activate=model_cfg['msk_activate'], causal=model_cfg['causal'])model.eval()# 检查是否有预训练权重pretrained_file = os.path.join('pretrained', 'best_model.pth')if os.path.exists(pretrained_file):    checkpoint = torch.load(pretrained_file, map_location='cpu')    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:        model.load_state_dict(checkpoint['model_state_dict'])    else:        model.load_state_dict(checkpoint)    print("加载预训练权重成功")else:    print("无预训练权重，使用随机初始化")# 分析不同SNR下的输出特性fig, axes = plt.subplots(2, 4, figsize=(16, 8))for idx, snr in enumerate(snr_levels):    mix = generate_test_signal(snr_db=snr)    mix_tensor = torch.tensor(mix).unsqueeze(0).unsqueeze(0)  # (1, 1, T)    with torch.no_grad():        output = model(mix_tensor)    output_np = output[0].numpy()    # 绘制电极图    row = idx // 4    col = idx % 4    ax = axes[row][col]    im = ax.imshow(output_np, aspect='auto', origin='lower', cmap='hot',                   vmin=0, vmax=1)    ax.set_title("SNR=%ddB" % snr, fontsize=10)    if col == 0:        ax.set_ylabel("电极通道")    if row == 1:        ax.set_xlabel("时间帧")fig.suptitle("DeepACE 输出在不同SNR下的变化", fontsize=14)plt.tight_layout()plt.show()# 统计分析print("不同SNR下的输出统计:")print("SNR(dB)  均值    标准差   激活比例(>0.1)")print("-" * 45)for snr in snr_levels:    mix = generate_test_signal(snr_db=snr)    mix_tensor = torch.tensor(mix).unsqueeze(0).unsqueeze(0)    with torch.no_grad():        output = model(mix_tensor)    out = output[0].numpy()    print("%4d     %.4f  %.4f  %.2f%%" % (        snr, out.mean(), out.std(), np.mean(out > 0.1) * 100))

## §5 综合实验：修改方案 vs 基线将前面的修改组合起来，做一个完整的对比实验。

In [ ]:
# 综合对比实验框架def run_experiment(model, loss_fn, train_loader, valid_loader,                   n_epochs=5, lr=1e-3, device='cpu'):    """运行一个完整的训练实验，返回训练曲线"""    model = model.to(device)    optimizer = torch.optim.Adam(model.parameters(), lr=lr)    train_losses = []    valid_losses = []    for epoch in range(n_epochs):        # Train        model.train()        epoch_loss = 0        n_batch = 0        for mix, tgt in train_loader:            mix, tgt = mix.to(device), tgt.to(device)            optimizer.zero_grad()            out = model(mix)            min_len = min(out.shape[2], tgt.shape[2])            loss = loss_fn(out[:, :, :min_len], tgt[:, :, :min_len])            loss.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)            optimizer.step()            epoch_loss += loss.item()            n_batch += 1        train_losses.append(epoch_loss / max(n_batch, 1))        # Validate        model.eval()        val_loss = 0        val_batch = 0        with torch.no_grad():            for mix, tgt in valid_loader:                mix, tgt = mix.to(device), tgt.to(device)                out = model(mix)                min_len = min(out.shape[2], tgt.shape[2])                loss = loss_fn(out[:, :, :min_len], tgt[:, :, :min_len])                val_loss += loss.item()                val_batch += 1        valid_losses.append(val_loss / max(val_batch, 1))    return train_losses, valid_losses# 检查数据集data_dir = os.path.join(DEEPACE_DIR, 'data')train_mix_dir = os.path.join(data_dir, 'train', 'mixture')has_data = os.path.exists(train_mix_dir) and len(os.listdir(train_mix_dir)) > 0if has_data:    print("检测到mini数据集，运行对比实验...")    import torch.utils.data as data_utils    from dataset import Dataset, collate_fn    train_dataset = Dataset(        os.path.join(data_dir, 'train', 'mixture'),        os.path.join(data_dir, 'train', 'targetLGF'),        sample_rate=config['sample_rate'],        segment_length=config['segment_length'],        stim_rate=config['stim_rate']    )    valid_dataset = Dataset(        os.path.join(data_dir, 'valid', 'mixture'),        os.path.join(data_dir, 'valid', 'targetLGF'),        sample_rate=config['sample_rate'],        segment_length=config['segment_length'],        stim_rate=config['stim_rate']    )    train_loader = data_utils.DataLoader(train_dataset, batch_size=4,                                          shuffle=True, collate_fn=collate_fn)    valid_loader = data_utils.DataLoader(valid_dataset, batch_size=4,                                          shuffle=False, collate_fn=collate_fn)else:    print("无mini数据集，跳过对比实验")    print("请先运行: cd .. && python scripts/prepare_mini_dataset.py")

In [ ]:
if has_data:    device = 'cuda' if torch.cuda.is_available() else 'cpu'    # 实验1: 基线 (M=22, MSE)    model_baseline = DeepACE(        L=model_cfg['L'], N=model_cfg['N'], P=model_cfg['P'],        B=model_cfg['B'], S=model_cfg['S'], H=model_cfg['H'],        X=model_cfg['X'], R=model_cfg['R'], M=22,        msk_activate=model_cfg['msk_activate'], causal=model_cfg['causal']    )    print("实验1: 基线 (M=22, MSE)...")    tl1, vl1 = run_experiment(model_baseline, nn.MSELoss(),                              train_loader, valid_loader, n_epochs=5, device=device)    # 实验2: 少通道 (M=12, MSE)    model_m12 = DeepACE(        L=model_cfg['L'], N=model_cfg['N'], P=model_cfg['P'],        B=model_cfg['B'], S=model_cfg['S'], H=model_cfg['H'],        X=model_cfg['X'], R=model_cfg['R'], M=12,        msk_activate=model_cfg['msk_activate'], causal=model_cfg['causal']    )    # 需要调整target的通道数    print("实验2: M=12 (MSE)...")    tl2, vl2 = run_experiment(model_m12, nn.MSELoss(),                              train_loader, valid_loader, n_epochs=5, device=device)    # 实验3: 稀疏损失    model_sparse = DeepACE(        L=model_cfg['L'], N=model_cfg['N'], P=model_cfg['P'],        B=model_cfg['B'], S=model_cfg['S'], H=model_cfg['H'],        X=model_cfg['X'], R=model_cfg['R'], M=22,        msk_activate=model_cfg['msk_activate'], causal=model_cfg['causal']    )    print("实验3: M=22 + SparseMSE...")    tl3, vl3 = run_experiment(model_sparse, SparseMSE(0.05),                              train_loader, valid_loader, n_epochs=5, device=device)    # 绘制对比图    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))    epochs = range(1, 6)    ax1.plot(epochs, tl1, 'o-', label='Baseline (M=22, MSE)')    ax1.plot(epochs, tl2, 's-', label='M=12, MSE')    ax1.plot(epochs, tl3, '^-', label='M=22, SparseMSE')    ax1.set_title('Train Loss')    ax1.set_xlabel('Epoch')    ax1.set_ylabel('Loss')    ax1.legend()    ax1.grid(True, alpha=0.3)    ax2.plot(epochs, vl1, 'o-', label='Baseline (M=22, MSE)')    ax2.plot(epochs, vl2, 's-', label='M=12, MSE')    ax2.plot(epochs, vl3, '^-', label='M=22, SparseMSE')    ax2.set_title('Valid Loss')    ax2.set_xlabel('Epoch')    ax2.set_ylabel('Loss')    ax2.legend()    ax2.grid(True, alpha=0.3)    plt.tight_layout()    plt.show()    print()    print("最终验证Loss对比:")    print("  Baseline: %.6f" % vl1[-1])    print("  M=12:     %.6f" % vl2[-1])    print("  Sparse:   %.6f" % vl3[-1])else:    print("跳过对比实验（无数据集）")    print()    print("如果你有数据集，可以尝试以下实验设计：")    print("1. 改变M(输出通道数)并观察训练曲线变化")    print("2. 使用不同的损失函数对比收敛速度")    print("3. 添加/移除SE Layer观察效果")

## §6 实验报告模板完成修改实验后，请按以下模板撰写简短报告：### 实验报告模板**1. 实验目的**（1-2句话）**2. 假设**（1-2句话）**3. 修改内容**- 修改了哪些文件/参数？- 修改的具体代码**4. 实验设置**- 数据集：mini数据集（X条训练，Y条验证）- 训练参数：epoch, batch_size, lr- 对比基线：原始模型**5. 结果**- 训练/验证Loss曲线- 定量对比表格**6. 分析**- 结果是否支持假设？- 为什么？（可能的解释）- 如果不支持，下一步可以尝试什么？**7. 结论**（1-2句话）

## §7 从"读懂代码"到"改进模型"的方法总结本次课的方法论：1. **观察现象**：运行原始代码，观察输出特征2. **提出问题**：哪些地方可以改进？为什么？3. **查阅文献**：这个问题别人是怎么解决的？4. **设计实验**：明确假设 -> 修改 -> 对比 -> 分析5. **迭代改进**：基于结果调整方向> 这正是科研工作的流程！模块4的目的是让学生从"跟着教程做"过渡到"独立分析和改进"。

## §8 推荐的后续研究方向基于DeepACE，以下是一些值得探索的方向：1. **自适应通道选择**：让模型根据输入动态决定激活几个通道（而非固定N-of-M）2. **多任务学习**：同时预测电极图和语音增强后的语谱图3. **说话人自适应**：针对不同说话人微调模型4. **实时推理优化**：将模型部署到嵌入式设备（CI处理器）5. **主观评估**：将模型输出转换为声信号，让CI用户试听评估6. **跨数据集泛化**：在不同噪声类型和说话人上测试泛化性能

---## 小结本节课我们：1. 学习了实验设计方法论（假设->修改->对比->评估->分析）2. 尝试了三种修改方向：通道选择策略、损失函数、噪声鲁棒性3. 实践了完整的对比实验流程**课后综合任务**：- 基础：运行DeepACE的完整训练和推理流程，提交运行记录和输出分析- 进阶：设计并实施一个修改实验，对比修改前后的效果差异，写一份简短实验报告